In [ ]:
# Run folder: outputs go to runs/08_rq2_normalised_0_1/
import os
if "REPO_ROOT" not in globals():
    REPO_ROOT = os.path.abspath("..")
RUN_DIR = os.path.join(REPO_ROOT, "runs", "08_rq2_normalised_0_1")
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(RUN_DIR)

In [ ]:
# Install requirements
%pip install -r ../../requirements.txt

In [ ]:
# Imports
import os
import json
import shutil
import re
import sys
from collections import OrderedDict

import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, ConcatDataset

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold

import torchaudio
from transformers import AutoConfig, AutoTokenizer, DebertaV2Model, Wav2Vec2FeatureExtractor, Wav2Vec2Model

import librosa
import parselmouth
from scipy.signal import find_peaks

import whisper
import glob
import matplotlib.pyplot as plt

In [ ]:
# Configuration
DATA_ROOT = "../../data/ADReSSo_RQ2"

SOURCE = "train"
TARGET = "test"

CORPORA = {
    "train": {
        "name": "Picture description (unstructured)",
        "ad_wav_dir": os.path.join(DATA_ROOT, "train_unstructured", "ad"),
        "cn_wav_dir": os.path.join(DATA_ROOT, "train_unstructured", "cn"),
        "metadata_csv": os.path.join(DATA_ROOT, "train_metadata.csv"),
        "neg_name": "CN",
        "expected": {"ad": 122, "cn": 115},
        "whisper_language": "en",
        "cache": "train_unstructured",
    },
    "test": {
        "name": "Verbal fluency (structured)",
        "ad_wav_dir": os.path.join(DATA_ROOT, "test_structured", "ad"),
        "cn_wav_dir": os.path.join(DATA_ROOT, "test_structured", "cn"),
        "metadata_csv": os.path.join(DATA_ROOT, "test_metadata.csv"),
        "neg_name": "CN",
        "expected": {"ad": 59, "cn": 10},
        "whisper_language": "en",
        "cache": "test_structured",
    },
}
CLASS_LABEL = {"ad": 1, "cn": 0}

WHISPER_MODEL_SIZE = "medium"
TEXT_EMBEDDING_MODEL = "microsoft/deberta-v3-base"
WAV2VEC_MODEL = "facebook/wav2vec2-base-960h"
NUM_EMBEDDING_FEATURES = AutoConfig.from_pretrained(WAV2VEC_MODEL).hidden_size
NUM_MFCC_FEATURES = 47
AUDIO_CHANNELS = 1
CUDA = True

EXTRACTION_ROOT = "./extraction"
FORCE_REEXTRACT = False

NORM_RANGE = (0, 1)
NORMALIZED_AUDIO_ROOT = "./normalized_audio"
FORCE_RENORMALIZE = False


def corpus_paths(key):
    raw = os.path.join(EXTRACTION_ROOT, CORPORA[key]["cache"])
    scaled = os.path.join("./preprocessing", CORPORA[key]["cache"])
    p = {}
    for cls in CLASS_LABEL:
        p[f"{cls}_transcribed_dir"] = os.path.join(raw, "transcripts", cls)
        p[f"{cls}_text_dir"] = os.path.join(raw, "processed_text", cls)
        p[f"{cls}_raw_features_csv"] = os.path.join(raw, f"audio_features_{cls}_raw.csv")
        p[f"{cls}_raw_embeddings_csv"] = os.path.join(raw, f"audio_embeddings_{cls}_raw.csv")
        p[f"{cls}_features_dir"] = os.path.join(scaled, cls)
        p[f"{cls}_csv"] = os.path.join(scaled, cls, f"audio_features_{cls}_raw.csv")
        p[f"{cls}_embedding_csv"] = os.path.join(scaled, cls, f"audio_embeddings_{cls}.csv")
    return p

PATHS = {key: corpus_paths(key) for key in CORPORA}

AUDIO_FEATURES_SCALER_PATH = "./preprocessing/scaler_audio_features_train.pkl"
AUDIO_EMBEDDING_SCALER_PATH = "./preprocessing/scaler_audio_emb_train.pkl"

BATCH_SIZE = 8
EPOCHS = 200
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
NUM_FOLDS = 5
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 10

FINAL_EPOCHS_OVERRIDE = None

RESULTS_DIR = "./results_final"
SAVE_MODEL_PATH = os.path.join(RESULTS_DIR, "model")
FINAL_MODEL_PATH = os.path.join(RESULTS_DIR, "final_model.pth")
LOG_PATH = os.path.join(RESULTS_DIR, "training.log")
FINAL_LOG_PATH = os.path.join(RESULTS_DIR, "final_training.log")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("./preprocessing", exist_ok=True)

print(f"RQ2: train on {CORPORA[SOURCE]['name']} -> test on {CORPORA[TARGET]['name']}")
print(f"Wav2Vec2 embedding size: {NUM_EMBEDDING_FEATURES}")

In [ ]:
# Reproducibility
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Per-class file lists and metadata
def list_wavs(d):
    if not os.path.isdir(d):
        raise FileNotFoundError(f"{d} does not exist (check the paths in the configuration)")
    return {os.path.splitext(f)[0]: os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith('.wav')}

def corpus_wav_files(cfg):
    ad, cn = list_wavs(cfg["ad_wav_dir"]), list_wavs(cfg["cn_wav_dir"])
    both = sorted(set(ad) & set(cn))
    assert not both, f"{cfg['name']}: {len(both)} file names occur in both ad/ and cn/, e.g. {both[:5]}"
    return {"ad": list(ad.values()), "cn": list(cn.values())}

WAV_FILES = {key: corpus_wav_files(cfg) for key, cfg in CORPORA.items()}

def normalize_column(c):
    return re.sub(r'[^0-9a-z]+', '_', c.strip().lower()).strip('_')

def find_column(df, candidates):
    return next((c for c in candidates if df is not None and c in df.columns), None)

def read_metadata(csv_path):
    if not os.path.exists(csv_path):
        print(f"WARNING {csv_path} not found -> metadata checks and the subgroup analysis are skipped")
        return None
    meta = pd.read_csv(csv_path, dtype=str)
    meta.columns = [normalize_column(c) for c in meta.columns]
    id_col = find_column(meta, ['id', 'patient_id', 'file', 'filename']) or meta.columns[0]
    meta = meta.rename(columns={id_col: 'patient_id'})
    meta = meta.apply(lambda col: col.str.strip())
    meta['patient_id'] = meta['patient_id'].map(lambda x: os.path.splitext(x)[0] if x.lower().endswith('.wav') else x)
    dup = sorted(meta.loc[meta['patient_id'].duplicated(), 'patient_id'])
    assert not dup, f"{csv_path}: IDs listed more than once: {dup[:5]}"
    return meta

AD_LABEL_VALUES = {'ad', '1', 'dementia', 'probablead', 'possiblead', 'probable_ad', 'possible_ad'}
NEG_LABEL_VALUES = {'cn', '0', 'control', 'hc', 'mci', 'nc'}

METADATA = {}
for key, cfg in CORPORA.items():
    meta = read_metadata(cfg["metadata_csv"])
    METADATA[key] = meta
    if meta is None:
        continue
    folder_label = {os.path.splitext(os.path.basename(f))[0]: CLASS_LABEL[cls]
                    for cls in WAV_FILES[key] for f in WAV_FILES[key][cls]}
    no_meta = sorted(set(folder_label) - set(meta['patient_id']))
    if no_meta:
        print(f"WARNING {cfg['name']}: {len(no_meta)} recordings have no row in {os.path.basename(cfg['metadata_csv'])}, e.g. {no_meta[:5]}")
    no_audio = sorted(set(meta['patient_id']) - set(folder_label))
    if no_audio:
        print(f"WARNING {cfg['name']}: {len(no_audio)} metadata rows have no .wav, e.g. {no_audio[:5]}")
    if 'label' in meta.columns:
        values = meta['label'].str.lower()
        if set(values.dropna()) <= AD_LABEL_VALUES | NEG_LABEL_VALUES:
            meta_label = dict(zip(meta['patient_id'], values.isin(AD_LABEL_VALUES).astype(int)))
            wrong = sorted(i for i in folder_label if i in meta_label and meta_label[i] != folder_label[i])
            assert not wrong, f"{cfg['name']}: metadata label differs from the folder for {wrong[:10]}"
            print(f"{cfg['name']}: metadata label agrees with the folder for all {len(set(folder_label) & set(meta_label))} recordings")
        else:
            print(f"NOTE {cfg['name']}: label values {sorted(set(values.dropna()))} not recognised -> label check skipped")
    print(f"{cfg['name']}: metadata columns {list(meta.columns)}")

for key, cfg in CORPORA.items():
    n_ad, n_cn = len(WAV_FILES[key]["ad"]), len(WAV_FILES[key]["cn"])
    if n_ad == 0 or n_cn == 0:
        raise ValueError(f"{cfg['name']}: no recordings for {'AD' if n_ad == 0 else cfg['neg_name']}")
    exp = cfg["expected"]
    if (n_ad, n_cn) != (exp["ad"], exp["cn"]):
        print(f"WARNING {cfg['name']}: expected AD={exp['ad']}, {cfg['neg_name']}={exp['cn']} but found AD={n_ad}, {cfg['neg_name']}={n_cn}")
    role = "TRAIN" if key == SOURCE else "TEST"
    print(f"[{role}] {cfg['name']}: AD={n_ad}, {cfg['neg_name']}={n_cn}, total={n_ad + n_cn}")

In [ ]:
# Min–max normalization of every audio file to [0, 1]
import soundfile as sf

NORM_LO, NORM_HI = NORM_RANGE

def minmax_normalize_wav(in_path, out_path):
    audio, sr = sf.read(in_path, dtype='float64', always_2d=True)
    lo, hi = float(audio.min()), float(audio.max())
    if hi > lo:
        norm = NORM_LO + (NORM_HI - NORM_LO) * (audio - lo) / (hi - lo)
    else:
        print(f"WARNING {in_path} is constant (min = max = {lo}) -> written at the middle of the range")
        norm = np.full_like(audio, (NORM_LO + NORM_HI) / 2)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    sf.write(out_path, norm.astype(np.float32), sr, subtype='FLOAT')
    return {'orig_min': lo, 'orig_max': hi, 'sample_rate': sr,
            'channels': audio.shape[1], 'n_samples': audio.shape[0]}

RAW_WAV_FILES = WAV_FILES
NORM_WAV_FILES = {}
log_rows = []
for key, cfg in CORPORA.items():
    NORM_WAV_FILES[key] = {}
    for cls, files in RAW_WAV_FILES[key].items():
        out_dir = os.path.join(NORMALIZED_AUDIO_ROOT, cfg["cache"], cls)
        NORM_WAV_FILES[key][cls] = []
        for in_path in tqdm(files, desc=f"Min-max {key}/{cls}"):
            out_path = os.path.join(out_dir, os.path.basename(in_path))
            if FORCE_RENORMALIZE or not os.path.exists(out_path):
                info = minmax_normalize_wav(in_path, out_path)
            else:
                a, sr = sf.read(in_path, dtype='float64', always_2d=True)
                info = {'orig_min': float(a.min()), 'orig_max': float(a.max()), 'sample_rate': sr,
                        'channels': a.shape[1], 'n_samples': a.shape[0]}
            chk, chk_sr = sf.read(out_path, dtype='float64', always_2d=True)
            assert chk.shape == (info['n_samples'], info['channels']) and chk_sr == info['sample_rate'], \
                f"{out_path}: length / channels / sample rate changed"
            if info['orig_max'] > info['orig_min']:
                assert abs(chk.min() - NORM_LO) < 1e-6 and abs(chk.max() - NORM_HI) < 1e-6, \
                    f"{out_path}: range [{chk.min()}, {chk.max()}] is not {list(NORM_RANGE)} (stale file? set FORCE_RENORMALIZE = True)"
            NORM_WAV_FILES[key][cls].append(out_path)
            log_rows.append({'set': key, 'class': cls,
                             'patient_id': os.path.splitext(os.path.basename(in_path))[0],
                             **info, 'norm_min': float(chk.min()), 'norm_max': float(chk.max()),
                             'norm_mean': float(chk.mean()),
                             'original_path': in_path, 'normalized_path': out_path})

minmax_log = pd.DataFrame(log_rows)
os.makedirs(NORMALIZED_AUDIO_ROOT, exist_ok=True)
minmax_log.to_csv(os.path.join(NORMALIZED_AUDIO_ROOT, "minmax_log.csv"), index=False)

WAV_FILES = NORM_WAV_FILES
for key, cfg in CORPORA.items():
    sub = minmax_log[minmax_log['set'] == key]
    print(f"{cfg['name']}: {len(sub)} files normalized to [{sub['norm_min'].min():.3f}, {sub['norm_max'].max():.3f}] "
          f"(original min {sub['orig_min'].min():.3f}..{sub['orig_min'].max():.3f}, "
          f"max {sub['orig_max'].min():.3f}..{sub['orig_max'].max():.3f}; "
          f"mean after normalization {sub['norm_mean'].median():.3f} (median); "
          f"sample rates {sorted(sub['sample_rate'].unique())})")
print(f"Saved {os.path.join(NORMALIZED_AUDIO_ROOT, 'minmax_log.csv')}")

In [ ]:
# Transcription
def transcribe_audio_files(audio_paths, model_size="base", output_dir=None, language=None):
    model = whisper.load_model(model_size)
    os.makedirs(output_dir, exist_ok=True)

    for audio_path in audio_paths:
        output_file = os.path.join(output_dir, os.path.splitext(os.path.basename(audio_path))[0] + '.txt')
        print(f"Transcribing {audio_path}...")
        result = model.transcribe(audio_path, language=language, task="transcribe")
        text = result["text"]
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Saved transcription to {output_file}")

In [ ]:
# Acoustic features
def extract_features(audio_path, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1):
    y, sr = librosa.load(audio_path, sr=sr)
    duration = librosa.get_duration(y=y, sr=sr)
    energy = np.array([
        sum(abs(y[i:i+frame_length]**2))
        for i in range(0, len(y), hop_length)
    ])
    threshold = np.percentile(energy, silence_threshold * 100)
    silence_frames = energy < threshold
    speech_frames = ~silence_frames
    total_speech_frames = np.sum(speech_frames)
    total_speech_time = total_speech_frames * hop_length / sr
    silence_durations = []
    current_silence = 0
    for is_silence in silence_frames:
        if is_silence:
            current_silence += 1
        elif current_silence > 0:
            silence_durations.append(current_silence * hop_length / sr)
            current_silence = 0
    num_pauses = len(silence_durations)
    total_pause_duration = sum(silence_durations)
    avg_pause_duration = np.mean(silence_durations) if silence_durations else 0
    max_pause_duration = max(silence_durations) if silence_durations else 0
    pause_duration_std = np.std(silence_durations) if silence_durations else 0
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    pitches = pitches.flatten()
    magnitudes = magnitudes.flatten()
    pitches = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
    pitch_std = np.std(pitches) if len(pitches) > 0 else 0
    pitch_range = np.ptp(pitches) if len(pitches) > 0 else 0
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    intensity = librosa.amplitude_to_db(S, ref=np.max)
    intensity_mean = np.mean(intensity)
    intensity_std = np.std(intensity)
    intensity_range = np.ptp(intensity)

    spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)[0]
    spectral_centroid_mean = np.mean(spectral_centroids)
    spectral_centroid_std = np.std(spectral_centroids)

    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=frame_length, hop_length=hop_length)
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    articulation_rate = total_speech_time / (duration - total_pause_duration) if (duration - total_pause_duration) > 0 else 0

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    peaks = find_peaks(onset_env, height=np.mean(onset_env))[0]
    estimated_syllables = len(peaks)
    speaking_rate = estimated_syllables / total_speech_time if total_speech_time > 0 else 0

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch(time_step=hop_length/sr)
    point_process = parselmouth.praat.call([snd, pitch], "To PointProcess (cc)")
    local_jitter = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3)
    local_shimmer = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3, 1.6)
    harmonicity = snd.to_harmonicity_cc(time_step=hop_length/sr)
    hnr_values = harmonicity.values
    hnr_mean = np.mean(hnr_values[hnr_values != 0]) if np.any(hnr_values != 0) else 0

    formant = snd.to_formant_burg(time_step=hop_length/sr)
    f1 = []
    f2 = []
    f3 = []
    for t in np.arange(0, snd.duration, hop_length / sr):
        try:
            f1.append(formant.get_value_at_time(1, t))
            f2.append(formant.get_value_at_time(2, t))
            f3.append(formant.get_value_at_time(3, t))
        except:
            continue
    f1 = np.array(f1)
    f2 = np.array(f2)
    f3 = np.array(f3)
    formant_1_mean = np.mean(f1) if len(f1) > 0 else 0
    formant_1_std = np.std(f1) if len(f1) > 0 else 0
    formant_2_mean = np.mean(f2) if len(f2) > 0 else 0
    formant_2_std = np.std(f2) if len(f2) > 0 else 0
    formant_3_mean = np.mean(f3) if len(f3) > 0 else 0
    formant_3_std = np.std(f3) if len(f3) > 0 else 0

    features = {
        'duration': duration,
        'total_speech_time': total_speech_time,
        'speech_pause_ratio': total_speech_time / duration if duration > 0 else 0,
        'num_pauses': num_pauses,
        'total_pause_duration': total_pause_duration,
        'avg_pause_duration': avg_pause_duration,
        'max_pause_duration': max_pause_duration,
        'pause_duration_std': pause_duration_std,
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'pitch_range': pitch_range,
        'intensity_mean': intensity_mean,
        'intensity_std': intensity_std,
        'intensity_range': intensity_range,
        'articulation_rate': articulation_rate,
        'speaking_rate': speaking_rate,
        'spectral_centroid_mean': spectral_centroid_mean,
        'spectral_centroid_std': spectral_centroid_std,
        'zcr_mean': zcr_mean,
        'zcr_std': zcr_std,
        'jitter_local': local_jitter,
        'shimmer_local': local_shimmer,
        'hnr_mean': hnr_mean,
        'formant_1_mean': formant_1_mean,
        'formant_1_std': formant_1_std,
        'formant_2_mean': formant_2_mean,
        'formant_2_std': formant_2_std,
        'formant_3_mean': formant_3_mean,
        'formant_3_std': formant_3_std,
    }

    for i, (mean, std) in enumerate(zip(mfccs_mean, mfccs_std)):
        features[f'mfcc_{i+1}_mean'] = mean
        features[f'mfcc_{i+1}_std'] = std

    return features

def process_audio_feature_files(audio_paths, cls, output_csv, sr, frame_length, hop_length, silence_threshold):
    all_features = []
    for audio_path in audio_paths:
        patient_id = os.path.splitext(os.path.basename(audio_path))[0]
        print(f"Processing {audio_path}...")
        try:
            features = extract_features(
                audio_path,
                sr=sr,
                frame_length=frame_length,
                hop_length=hop_length,
                silence_threshold=silence_threshold
            )
            features['patient_id'] = patient_id
            features['class'] = cls
            all_features.append(features)
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    df = pd.DataFrame(all_features)
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

In [ ]:
# Wav2Vec2 embeddings
def extract_embeddings(audio_path, model, processor, device):
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0)
        print(f"Converted stereo to mono for {audio_path}")
    else:
        speech_array = speech_array.squeeze(0)

    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)
        sampling_rate = 16000

    speech = speech_array.numpy()

    inputs = processor(speech, sampling_rate=sampling_rate, return_tensors="pt", padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state
    embedding = embeddings.mean(dim=1).squeeze().cpu().numpy()

    return embedding

def process_audio_embedding_files(audio_paths, output_csv, model_name=WAV2VEC_MODEL):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    model.to(device)
    model.eval()

    all_embeddings = []
    for audio_path in tqdm(audio_paths, desc=f"Embedding {os.path.dirname(output_csv)}"):
        patient_id = os.path.splitext(os.path.basename(audio_path))[0]
        try:
            embedding = extract_embeddings(audio_path, model, processor, device)
            all_embeddings.append({'patient_id': patient_id, 'embedding': embedding})
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            continue

    if len(all_embeddings) == 0:
        print("No embeddings were extracted. Please check for errors.")
        return

    df = pd.DataFrame(all_embeddings)
    embedding_cols = pd.DataFrame(df['embedding'].tolist())
    embedding_cols['patient_id'] = df['patient_id']
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    embedding_cols.to_csv(output_csv, index=False)
    print(f"Embeddings saved to {output_csv}")

In [ ]:
# Run data extraction (both sets)
def needs_run(output_path):
    if FORCE_REEXTRACT:
        return True
    if output_path.endswith('.csv'):
        return not os.path.exists(output_path)
    return not (os.path.isdir(output_path) and any(f.endswith('.txt') for f in os.listdir(output_path)))

for key, cfg in CORPORA.items():
    p = PATHS[key]
    for cls in WAV_FILES[key]:
        wav_files = WAV_FILES[key][cls]

        if needs_run(p[f"{cls}_transcribed_dir"]):
            transcribe_audio_files(wav_files, model_size=WHISPER_MODEL_SIZE,
                                   output_dir=p[f"{cls}_transcribed_dir"],
                                   language=cfg["whisper_language"])
        else:
            print(f"[skip] transcripts {key}/{cls}")

        if needs_run(p[f"{cls}_raw_features_csv"]):
            process_audio_feature_files(wav_files, cls, p[f"{cls}_raw_features_csv"],
                                        sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1)
        else:
            print(f"[skip] acoustic features {key}/{cls}")

        if needs_run(p[f"{cls}_raw_embeddings_csv"]):
            process_audio_embedding_files(wav_files, p[f"{cls}_raw_embeddings_csv"])
        else:
            print(f"[skip] wav2vec2 embeddings {key}/{cls}")

In [ ]:
# Text cleaning
def preprocess_text(text):
    text = text.lower()
    text = ' '.join(text.split())
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)

    return text

def process_text_files(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.txt'):
            input_file = os.path.join(input_dir, filename)
            output_file = os.path.join(output_dir, filename)
            with open(input_file, 'r', encoding='utf-8') as f:
                text = f.read()
            processed_text = preprocess_text(text)
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(processed_text)

            print(f"Processed {filename}")

In [ ]:
# Audio feature scaling
def preprocess_csv(input_path, output_path, scaler_path):
    df = pd.read_csv(input_path, dtype={'patient_id': str})
    df = df.drop(['jitter_local', 'shimmer_local', 'formant_1_mean',
                  'formant_1_std', 'formant_2_mean', 'formant_2_std',
                  'formant_3_mean', 'formant_3_std', 'class'], axis=1)

    patient_ids = df['patient_id']
    df = df.drop(['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled_features = scaler.transform(df)
    df_scaled = pd.DataFrame(scaled_features, columns=df.columns)
    df_scaled['patient_id'] = patient_ids
    output_file = os.path.join(output_path, os.path.basename(input_path))
    df_scaled.to_csv(output_file, index=False)
    print(f"Processed file saved at: {output_file}")

In [ ]:
# Audio embedding scaling
def process_csv(input_csv, scaler_path, output_csv):
    df = pd.read_csv(input_csv, dtype={'patient_id': str})
    patient_ids = df['patient_id']
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = joblib.load(scaler_path)
    standardized_features = scaler.transform(features)
    standardized_df = pd.DataFrame(standardized_features, columns=features.columns)
    standardized_df['patient_id'] = patient_ids
    standardized_df.to_csv(output_csv, index=False)
    print(f"Processed CSV saved to '{output_csv}'")

In [ ]:
# Fit scalers on the TRAINING corpus only
AUDIO_FEATURE_DROP_COLUMNS = ['jitter_local', 'shimmer_local', 'formant_1_mean',
                              'formant_1_std', 'formant_2_mean', 'formant_2_std',
                              'formant_3_mean', 'formant_3_std', 'class']

def fit_audio_feature_scaler(raw_csv_paths, scaler_path):
    df = pd.concat([pd.read_csv(p) for p in raw_csv_paths], ignore_index=True)
    df = df.drop(AUDIO_FEATURE_DROP_COLUMNS + ['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = StandardScaler().fit(df)
    joblib.dump(scaler, scaler_path)
    print(f"Fitted audio-feature scaler on {len(df)} recordings, {df.shape[1]} features -> {scaler_path}")

def fit_audio_embedding_scaler(raw_csv_paths, scaler_path):
    df = pd.concat([pd.read_csv(p) for p in raw_csv_paths], ignore_index=True)
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = StandardScaler().fit(features)
    joblib.dump(scaler, scaler_path)
    print(f"Fitted embedding scaler on {len(features)} recordings, {features.shape[1]} dims -> {scaler_path}")

In [ ]:
# Run data processing
src = PATHS[SOURCE]
fit_audio_feature_scaler([src[f"{cls}_raw_features_csv"] for cls in CLASS_LABEL], AUDIO_FEATURES_SCALER_PATH)
fit_audio_embedding_scaler([src[f"{cls}_raw_embeddings_csv"] for cls in CLASS_LABEL], AUDIO_EMBEDDING_SCALER_PATH)

for key in CORPORA:
    p = PATHS[key]
    for cls in WAV_FILES[key]:
        if needs_run(p[f"{cls}_text_dir"]):
            process_text_files(p[f"{cls}_transcribed_dir"], p[f"{cls}_text_dir"])
        else:
            print(f"[skip] text cleaning {key}/{cls}")

        os.makedirs(p[f"{cls}_features_dir"], exist_ok=True)
        preprocess_csv(p[f"{cls}_raw_features_csv"], p[f"{cls}_features_dir"], AUDIO_FEATURES_SCALER_PATH)
        process_csv(p[f"{cls}_raw_embeddings_csv"], AUDIO_EMBEDDING_SCALER_PATH, p[f"{cls}_embedding_csv"])

In [ ]:
# Check that all recordings were processed
for key, cfg in CORPORA.items():
    p = PATHS[key]
    for cls in WAV_FILES[key]:
        cls_name = {"ad": "AD", "cn": cfg["neg_name"]}[cls]
        wav_ids = {os.path.splitext(os.path.basename(f))[0] for f in WAV_FILES[key][cls]}
        feat_ids = set(pd.read_csv(p[f"{cls}_csv"], dtype={'patient_id': str})['patient_id'])
        emb_ids = set(pd.read_csv(p[f"{cls}_embedding_csv"], dtype={'patient_id': str})['patient_id'])
        text_ids = {os.path.splitext(f)[0] for f in os.listdir(p[f"{cls}_text_dir"]) if f.endswith('.txt')}
        missing = {name: sorted(wav_ids - ids) for name, ids in
                   [("features", feat_ids), ("embeddings", emb_ids), ("text", text_ids)] if wav_ids - ids}
        extra = (feat_ids | emb_ids) - wav_ids
        status = "OK" if not missing else f"MISSING {({k: len(v) for k, v in missing.items()})}"
        print(f"{cfg['name']} {cls_name}: {len(wav_ids)} wavs -> {len(wav_ids & feat_ids & emb_ids & text_ids)} usable  [{status}]")
        for name, ids in missing.items():
            print(f"    no {name}: {ids[:10]}{' ...' if len(ids) > 10 else ''}")
        if extra:
            print(f"    WARNING {len(extra)} cached IDs are not in this class anymore (stale ./extraction cache?), "
                  f"e.g. {sorted(extra)[:5]} -> set FORCE_REEXTRACT = True")

In [ ]:
# Dataset
class DementiaDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        label_value=None,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path, dtype={'patient_id': str})
        self.embedding_data = pd.read_csv(embedding_csv_path, dtype={'patient_id': str})
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length
        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        self.label_value = label_value
        self.data['label'] = self.label_value

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r', encoding='utf-8') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label

def create_full_dataset(
    ad_text_dir,
    cn_text_dir,
    ad_csv,
    cn_csv,
    ad_embedding_csv,
    cn_embedding_csv
):
    ad_dataset = DementiaDataset(
        ad_csv,
        ad_embedding_csv,
        ad_text_dir,
        label_value=1
    )
    cn_dataset = DementiaDataset(
        cn_csv,
        cn_embedding_csv,
        cn_text_dir,
        label_value=0
    )
    full_dataset = ConcatDataset([ad_dataset, cn_dataset])
    return full_dataset

In [ ]:
# Model — NeuroXVocal
class NeuroXVocal(nn.Module):
    def __init__(
        self,
        num_audio_features,
        num_embedding_features,
        text_embedding_model,
        hidden_size=768
    ):
        super(NeuroXVocal, self).__init__()

        self.text_model = DebertaV2Model.from_pretrained(text_embedding_model)
        self.hidden_size = self.text_model.config.hidden_size
        self.audio_fc = nn.Sequential(
            nn.Linear(num_audio_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        self.embedding_fc = nn.Sequential(
            nn.Linear(num_embedding_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=8,
            dropout=0.35,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(self.hidden_size // 2, 1)
        )

    def forward(self, text_input, audio_input, embedding_input):
        input_ids = text_input['input_ids'].squeeze(1)
        attention_mask = text_input['attention_mask'].squeeze(1)
        text_outputs = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_embeddings = text_outputs.last_hidden_state
        audio_embeddings = self.audio_fc(audio_input)
        audio_embeddings = audio_embeddings.unsqueeze(1)
        embedding_embeddings = self.embedding_fc(embedding_input)
        embedding_embeddings = embedding_embeddings.unsqueeze(1)
        combined_embeddings = torch.cat(
            (audio_embeddings, embedding_embeddings, text_embeddings),
            dim=1
        )
        combined_embeddings = combined_embeddings.permute(1, 0, 2)
        transformer_output = self.transformer_encoder(combined_embeddings)
        pooled_output = transformer_output[0]
        logits = self.classifier(pooled_output).squeeze(-1)

        return logits

    def reset_parameters(self):
        def reset_layer_parameters(layer):
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()
        self.audio_fc.apply(reset_layer_parameters)
        self.embedding_fc.apply(reset_layer_parameters)
        self.transformer_encoder.apply(reset_layer_parameters)
        self.classifier.apply(reset_layer_parameters)

In [ ]:
# Training loop
def make_cv_splits(labels, n_splits=NUM_FOLDS, seed=SEED):
    labels = np.asarray(labels).astype(int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.zeros(len(labels)), labels)]

def train_model(
    build_model,
    full_dataset,
    epochs,
    learning_rate,
    log_path,
    save_model_path,
    device,
    num_folds=5,
    save_best_model=False,
    labels=None,
    splits=None
):
    criterion = BCEWithLogitsLoss()

    if labels is not None:
        labels = np.array(labels)
    if splits is not None:
        split_iter = splits
    elif labels is not None:
        kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(np.zeros(len(labels)), labels)
    else:
        kfold = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(full_dataset)

    history = {}
    overall_best_val_loss = float('inf')
    overall_best_fold = None

    for fold, (train_indices, val_indices) in enumerate(split_iter):
        print(f'Fold {fold+1}/{num_folds}')

        if labels is not None:
            train_counts = np.bincount(labels[train_indices].astype(int), minlength=2)
            val_counts = np.bincount(labels[val_indices].astype(int), minlength=2)
            print(f"Fold {fold+1}: train CN/AD = {train_counts[0]}/{train_counts[1]}, val CN/AD = {val_counts[0]}/{val_counts[1]}")

        train_subset = torch.utils.data.Subset(full_dataset, train_indices)
        val_subset = torch.utils.data.Subset(full_dataset, val_indices)

        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED)
        )
        val_loader = torch.utils.data.DataLoader(
            val_subset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = build_model()
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model.to(device)

        optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=1e-4
        )
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )

        best_val_loss = float('inf')
        epochs_no_improve = 0
        history[fold] = {'train_loss': [], 'val_loss': [], 'lr': [], 'best_epoch': None}

        for epoch in range(epochs):
            history[fold]['lr'].append(optimizer.param_groups[0]['lr'])
            model.train()
            running_loss = 0.0
            all_train_outputs = []
            all_train_labels = []

            for (
                text_data,
                audio_data,
                embedding_data,
                label
            ) in tqdm(
                train_loader,
                desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Training"
            ):
                optimizer.zero_grad()
                text_data = {
                    key: value.to(device) for key, value in text_data.items()
                }
                audio_data = audio_data.to(device)
                embedding_data = embedding_data.to(device)
                label = label.to(device)

                outputs = model(text_data, audio_data, embedding_data)

                loss = criterion(outputs, label.float())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                all_train_outputs.extend(outputs.detach().cpu().numpy())
                all_train_labels.extend(label.cpu().numpy())

            train_loss = running_loss / len(train_loader)
            train_predictions = (np.array(all_train_outputs) > 0).astype(int)
            train_accuracy = accuracy_score(
                np.array(all_train_labels),
                train_predictions
            )
            train_report = classification_report(
                np.array(all_train_labels),
                train_predictions,
                target_names=['CN', 'AD'],
                digits=4
            )

            model.eval()
            val_running_loss = 0.0
            all_val_outputs = []
            all_val_labels = []
            with torch.no_grad():
                for (
                    text_data,
                    audio_data,
                    embedding_data,
                    label
                ) in tqdm(
                    val_loader,
                    desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Validation"
                ):
                    text_data = {
                        key: value.to(device) for key, value in text_data.items()
                    }
                    audio_data = audio_data.to(device)
                    embedding_data = embedding_data.to(device)
                    label = label.to(device)

                    outputs = model(text_data, audio_data, embedding_data)
                    loss = criterion(outputs, label.float())

                    val_running_loss += loss.item()
                    all_val_outputs.extend(outputs.detach().cpu().numpy())
                    all_val_labels.extend(label.cpu().numpy())

            val_loss = val_running_loss / len(val_loader)
            val_predictions = (np.array(all_val_outputs) > 0).astype(int)
            val_accuracy = accuracy_score(
                np.array(all_val_labels),
                val_predictions
            )
            val_report = classification_report(
                np.array(all_val_labels),
                val_predictions,
                target_names=['CN', 'AD'],
                digits=4
            )

            history[fold]['train_loss'].append(train_loss)
            history[fold]['val_loss'].append(val_loss)

            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                history[fold]['best_epoch'] = epoch + 1
                if save_best_model:
                    best_model_path = f"{save_model_path}_fold{fold+1}_best.pth"
                    if isinstance(model, torch.nn.DataParallel):
                        torch.save(model.module.state_dict(), best_model_path)
                    else:
                        torch.save(model.state_dict(), best_model_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping at epoch {epoch+1} for fold {fold+1}")
                    break

            with open(log_path, 'a') as f:
                f.write(
                    f"Fold {fold+1}, Epoch {epoch+1}, "
                    f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}\n"
                )
                f.write("Train Classification Report:\n")
                f.write(f"{train_report}\n")
                f.write("Validation Classification Report:\n")
                f.write(f"{val_report}\n")

            print(
                f"Fold {fold+1}, Epoch {epoch+1}, "
                f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
            )
            print("Train Classification Report:")
            print(train_report)
            print("Validation Classification Report:")
            print(val_report)

        print(f"Fold {fold+1}: best epoch {history[fold]['best_epoch']} (val loss {best_val_loss:.4f})")

        if best_val_loss < overall_best_val_loss:
            overall_best_val_loss = best_val_loss
            overall_best_fold = fold

        del model, optimizer, scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if save_best_model and overall_best_fold is not None:
        best_fold_checkpoint = f"{save_model_path}_fold{overall_best_fold+1}_best.pth"
        overall_best_checkpoint = f"{save_model_path}_best.pth"
        shutil.copy2(best_fold_checkpoint, overall_best_checkpoint)
        print(f"Overall best fold: {overall_best_fold+1} (val loss {overall_best_val_loss:.4f}) -> saved as {overall_best_checkpoint}")

    return history

In [ ]:
# Run training
print(f"Starting k-fold cross-validation training on {CORPORA[SOURCE]['name']}...")
with open(LOG_PATH, 'w') as f:
    pass

src = PATHS[SOURCE]
train_parts = [DementiaDataset(src[f"{cls}_csv"], src[f"{cls}_embedding_csv"], src[f"{cls}_text_dir"],
                               label_value=CLASS_LABEL[cls]) for cls in CLASS_LABEL]
full_dataset = ConcatDataset(train_parts)

device = torch.device('cuda' if torch.cuda.is_available() and CUDA else 'cpu')

def build_model():
    return NeuroXVocal(
        num_audio_features=NUM_MFCC_FEATURES,
        num_embedding_features=NUM_EMBEDDING_FEATURES,
        text_embedding_model=TEXT_EMBEDDING_MODEL
    )

labels = np.concatenate([ds.data['label'].values for ds in train_parts]).astype(int)
train_patient_ids = np.concatenate([ds.data['patient_id'].astype(str).values for ds in train_parts])

cv_splits = make_cv_splits(labels)
val_fold = np.zeros(len(labels), dtype=int)
for k, (_, va) in enumerate(cv_splits):
    val_fold[va] = k + 1
pd.DataFrame({'patient_id': train_patient_ids, 'label': labels, 'val_fold': val_fold}).to_csv(
    os.path.join(RESULTS_DIR, "cv_folds.csv"), index=False)

print(f"{len(full_dataset)} training recordings (AD {int(labels.sum())} / {CORPORA[SOURCE]['neg_name']} {int((labels == 0).sum())})")
for k, (tr, va) in enumerate(cv_splits):
    print(f"Fold {k+1}: train {len(tr)} (AD/CN {int(labels[tr].sum())}/{int((labels[tr] == 0).sum())}) | "
          f"val {len(va)} (AD/CN {int(labels[va].sum())}/{int((labels[va] == 0).sum())})")

history = train_model(
    build_model,
    full_dataset,
    EPOCHS,
    LEARNING_RATE,
    LOG_PATH,
    SAVE_MODEL_PATH,
    device,
    NUM_FOLDS,
    SAVE_BEST_MODEL,
    labels=labels,
    splits=cv_splits
)

In [ ]:
# Plot training and validation loss
n_folds = len(history)
fig, axes = plt.subplots(1, n_folds, figsize=(5 * n_folds, 4), sharey=True)
if n_folds == 1:
    axes = [axes]

for ax, fold in zip(axes, sorted(history.keys())):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    ax.plot(epochs_range, history[fold]['train_loss'], label='Train Loss')
    ax.plot(epochs_range, history[fold]['val_loss'], label='Val Loss')
    ax.set_title(f'Fold {fold + 1}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
per_fold_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_per_fold.png")
plt.savefig(per_fold_plot_path)
plt.show()
print(f"Saved plot to {per_fold_plot_path}")

In [ ]:
# Plot training and validation loss per fold
plt.figure(figsize=(8, 5))
for fold in sorted(history.keys()):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    plt.plot(epochs_range, history[fold]['train_loss'], linestyle='-', label=f'Fold {fold + 1} Train')
    plt.plot(epochs_range, history[fold]['val_loss'], linestyle='--', label=f'Fold {fold + 1} Val')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss (All Folds)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
overview_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_all_folds.png")
plt.savefig(overview_plot_path)
plt.show()
print(f"Saved plot to {overview_plot_path}")

In [ ]:
# Stage B recipe
fold_ids = sorted(history.keys())
best_epochs = [history[f]['best_epoch'] for f in fold_ids]
cv_final_epochs = int(np.round(np.median(best_epochs)))
FINAL_EPOCHS = FINAL_EPOCHS_OVERRIDE if FINAL_EPOCHS_OVERRIDE is not None else cv_final_epochs

def cv_lr_at(epoch_idx):
    return float(np.median([history[f]['lr'][min(epoch_idx, len(history[f]['lr']) - 1)] for f in fold_ids]))

FINAL_LR_SCHEDULE = [cv_lr_at(e) for e in range(FINAL_EPOCHS)]

recipe_df = pd.DataFrame({
    'fold': [f + 1 for f in fold_ids],
    'best_epoch': best_epochs,
    'epochs_run': [len(history[f]['train_loss']) for f in fold_ids],
    'best_val_loss': [min(history[f]['val_loss']) for f in fold_ids],
    'lr_at_best_epoch': [history[f]['lr'][history[f]['best_epoch'] - 1] for f in fold_ids],
})
print(recipe_df.to_string(index=False))
print()
print(f"Median best epoch: {cv_final_epochs}  ->  FINAL_EPOCHS = {FINAL_EPOCHS}"
      + ("  (override)" if FINAL_EPOCHS_OVERRIDE is not None else ""))
lr_changes = [e + 1 for e in range(1, FINAL_EPOCHS) if FINAL_LR_SCHEDULE[e] != FINAL_LR_SCHEDULE[e - 1]]
print(f"Final LR schedule: starts at {FINAL_LR_SCHEDULE[0]:.2e}, ends at {FINAL_LR_SCHEDULE[-1]:.2e}, "
      f"changes at epochs {lr_changes if lr_changes else 'none (constant)'}")

recipe = {
    'source': CORPORA[SOURCE]['name'],
    'cv_best_epochs': best_epochs,
    'cv_median_best_epoch': cv_final_epochs,
    'final_epochs': FINAL_EPOCHS,
    'final_epochs_override': FINAL_EPOCHS_OVERRIDE,
    'final_lr_schedule': FINAL_LR_SCHEDULE,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
}
recipe_path = os.path.join(RESULTS_DIR, "final_training_recipe.json")
with open(recipe_path, 'w') as f:
    json.dump(recipe, f, indent=2)
print(f"Saved {recipe_path}")

In [ ]:
# Stage B — final training on 100% of the training corpus
def train_final_model(build_model, dataset, lr_schedule, log_path, save_path, device):
    criterion = BCEWithLogitsLoss()

    torch.manual_seed(SEED)
    model = build_model()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr_schedule[0], weight_decay=1e-4)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    history = {'train_loss': [], 'train_accuracy': [], 'lr': []}
    with open(log_path, 'w') as f:
        pass

    n_epochs = len(lr_schedule)
    for epoch, lr in enumerate(lr_schedule):
        for group in optimizer.param_groups:
            group['lr'] = lr

        model.train()
        running_loss = 0.0
        all_outputs, all_labels = [], []
        for text_data, audio_data, embedding_data, label in tqdm(
            loader, desc=f"Final model, Epoch {epoch+1}/{n_epochs}"
        ):
            optimizer.zero_grad()
            text_data = {key: value.to(device) for key, value in text_data.items()}
            audio_data = audio_data.to(device)
            embedding_data = embedding_data.to(device)
            label = label.to(device)

            outputs = model(text_data, audio_data, embedding_data)
            loss = criterion(outputs, label.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            all_outputs.extend(outputs.detach().cpu().numpy())
            all_labels.extend(label.cpu().numpy())

        train_loss = running_loss / len(loader)
        train_predictions = (np.array(all_outputs) > 0).astype(int)
        train_accuracy = accuracy_score(np.array(all_labels), train_predictions)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['lr'].append(lr)

        msg = f"Final model, Epoch {epoch+1}/{n_epochs}, LR: {lr:.2e}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}"
        print(msg)
        with open(log_path, 'a') as f:
            f.write(msg + "\n")

    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state_dict, save_path)
    print(f"Saved final model (after epoch {n_epochs}) to {save_path}")

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return history

n_ad, n_cn = int(np.sum(labels)), int(len(labels) - np.sum(labels))
print(f"Final training on 100% of {CORPORA[SOURCE]['name']}: {len(full_dataset)} recordings "
      f"(AD {n_ad} / {CORPORA[SOURCE]['neg_name']} {n_cn}), {FINAL_EPOCHS} epochs")

final_history = train_final_model(
    build_model,
    full_dataset,
    FINAL_LR_SCHEDULE,
    FINAL_LOG_PATH,
    FINAL_MODEL_PATH,
    device
)

In [ ]:
# Plot the training curve of the final model
fig, ax1 = plt.subplots(figsize=(8, 4))
epochs_range = range(1, len(final_history['train_loss']) + 1)
ax1.plot(epochs_range, final_history['train_loss'], label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax2 = ax1.twinx()
ax2.plot(epochs_range, final_history['train_accuracy'], color='tab:orange', linestyle='--', label='Train Accuracy')
ax2.set_ylabel('Accuracy')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
ax1.set_title(f"Final model on 100% of {CORPORA[SOURCE]['name']} (no validation split)")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
final_plot_path = os.path.join(RESULTS_DIR, "final_model_training_curve.png")
plt.savefig(final_plot_path)
plt.show()
print(f"Saved plot to {final_plot_path}")

In [ ]:
# Evaluation helpers
class LabeledPatientDataset(DementiaDataset):
    def __getitem__(self, idx):
        text_tokens, audio_tensor, embedding_tensor, label = super().__getitem__(idx)
        return text_tokens, audio_tensor, embedding_tensor, label, self.data.iloc[idx]['patient_id']

def create_labeled_dataset(paths):
    return ConcatDataset([
        LabeledPatientDataset(paths["ad_csv"], paths["ad_embedding_csv"], paths["ad_text_dir"], label_value=1),
        LabeledPatientDataset(paths["cn_csv"], paths["cn_embedding_csv"], paths["cn_text_dir"], label_value=0),
    ])

def load_checkpoint(checkpoint_path, device):
    model = NeuroXVocal(
        num_audio_features=NUM_MFCC_FEATURES,
        num_embedding_features=NUM_EMBEDDING_FEATURES,
        text_embedding_model=TEXT_EMBEDDING_MODEL,
    )
    state_dict = torch.load(checkpoint_path, map_location=device)
    if 'module.' in list(state_dict.keys())[0]:
        state_dict = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

def predict(model, dataset, device, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    probs, labels_, ids = [], [], []
    with torch.no_grad():
        for text_data, audio_data, embedding_data, label, patient_id in loader:
            text_data = {key: value.to(device) for key, value in text_data.items()}
            logits = model(text_data, audio_data.to(device), embedding_data.to(device))
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels_.extend(label.numpy().astype(int).tolist())
            ids.extend(list(patient_id))
    return np.array(ids), np.array(labels_), np.array(probs)

def compute_metrics(y_true, probs, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    y_pred = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    both = len(np.unique(y_true)) == 2
    return {
        'n': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred) if both else float('nan'),
        'f1_ad': f1_score(y_true, y_pred, pos_label=1, zero_division=0) if (tp + fn) else float('nan'),
        'sensitivity': tp / (tp + fn) if (tp + fn) else float('nan'),
        'specificity': tn / (tn + fp) if (tn + fp) else float('nan'),
        'roc_auc': roc_auc_score(y_true, probs) if both else float('nan'),
        'pred_ad_rate': y_pred.mean(),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
    }

def fold_checkpoints():
    paths = [f"{SAVE_MODEL_PATH}_fold{k + 1}_best.pth" for k in range(NUM_FOLDS)]
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Missing fold checkpoints: {missing}")
    return paths

def freeze_model(model):
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)
    return model

device = torch.device('cuda' if torch.cuda.is_available() and CUDA else 'cpu')

CV_FOLDS_CSV = os.path.join(RESULTS_DIR, "cv_folds.csv")

def source_cv_splits(dataset):
    labels_ = np.concatenate([ds.data['label'].values for ds in dataset.datasets]).astype(int)
    ids_ = np.concatenate([ds.data['patient_id'].astype(str).values for ds in dataset.datasets])
    splits = make_cv_splits(labels_)
    if os.path.exists(CV_FOLDS_CSV):
        folds = pd.read_csv(CV_FOLDS_CSV, dtype={'patient_id': str})
        for k, (_, va) in enumerate(splits):
            trained = set(folds.loc[folds['val_fold'] == k + 1, 'patient_id'])
            assert set(ids_[va]) == trained, f"fold {k+1}: validation split differs from training"
    else:
        print(f"WARNING {CV_FOLDS_CSV} not found - cannot check the folds against training")
    return splits

NEG_NAME = {key: cfg['neg_name'] for key, cfg in CORPORA.items()}

def wilson_ci(k, n, z=1.959964):
    if n == 0:
        return (float('nan'), float('nan'))
    phat = k / n
    denom = 1 + z ** 2 / n
    center = (phat + z ** 2 / (2 * n)) / denom
    half = z * np.sqrt(phat * (1 - phat) / n + z ** 2 / (4 * n ** 2)) / denom
    return (float(center - half), float(center + half))

N_BOOTSTRAP = 2000

def bootstrap_ci(y_true, probs, n_boot=N_BOOTSTRAP, seed=SEED, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    pos, neg = np.flatnonzero(y_true == 1), np.flatnonzero(y_true == 0)
    if len(pos) == 0 or len(neg) == 0:
        return {'balanced_accuracy': (float('nan'),) * 2, 'roc_auc': (float('nan'),) * 2}
    rng = np.random.default_rng(seed)
    bal, auc = [], []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        y_b, p_b = y_true[idx], probs[idx]
        bal.append(balanced_accuracy_score(y_b, (p_b > threshold).astype(int)))
        auc.append(roc_auc_score(y_b, p_b))
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return {'balanced_accuracy': q(bal), 'roc_auc': q(auc)}

In [ ]:
# In-domain reference
source_dataset = create_labeled_dataset(PATHS[SOURCE])
source_labels = np.concatenate([ds.data['label'].values for ds in source_dataset.datasets]).astype(int)

oof_frames, source_fold_rows = [], []
for fold, ((_, val_indices), ckpt) in enumerate(zip(source_cv_splits(source_dataset), fold_checkpoints())):
    model = load_checkpoint(ckpt, device)
    ids, y, p = predict(model, torch.utils.data.Subset(source_dataset, val_indices), device)
    oof_frames.append(pd.DataFrame({'patient_id': ids, 'fold': fold + 1, 'true_label': y, 'predicted_probability': p}))
    source_fold_rows.append({'fold': fold + 1, **compute_metrics(y, p)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

source_oof_df = pd.concat(oof_frames, ignore_index=True)
source_oof_df['predicted_label'] = (source_oof_df['predicted_probability'] > 0.5).astype(int)
source_fold_df = pd.DataFrame(source_fold_rows)
source_oof_metrics = compute_metrics(source_oof_df['true_label'].values, source_oof_df['predicted_probability'].values)

source_oof_df.to_csv(os.path.join(RESULTS_DIR, f"train_oof_predictions.csv"), index=False)
source_fold_df.to_csv(os.path.join(RESULTS_DIR, f"train_per_fold_metrics.csv"), index=False)

print(f"In-domain ({CORPORA[SOURCE]['name']}) out-of-fold, pooled over {NUM_FOLDS} folds:")
print(pd.Series(source_oof_metrics).to_string())
source_fold_df

In [ ]:
# Cross-task transfer (RQ2 answer)
if not os.path.exists(FINAL_MODEL_PATH):
    raise FileNotFoundError(f"{FINAL_MODEL_PATH} not found - train the final model first")

target_dataset = create_labeled_dataset(PATHS[TARGET])

final_model = freeze_model(load_checkpoint(FINAL_MODEL_PATH, device))
n_trainable = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
assert n_trainable == 0 and not final_model.training, "final model is not frozen"
print(f"Final model frozen: {sum(p.numel() for p in final_model.parameters()):,} parameters, 0 trainable, eval mode")

target_ids, target_labels, final_probs = predict(final_model, target_dataset, device)
final_target_metrics = compute_metrics(target_labels, final_probs)
del final_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_pred_df = pd.DataFrame({
    'patient_id': target_ids,
    'true_label': target_labels,
    'predicted_probability': final_probs,
    'predicted_label': (final_probs > 0.5).astype(int),
})
final_pred_df['predicted_class'] = np.where(final_pred_df['predicted_label'] == 1, 'AD', 'CN')
if METADATA[TARGET] is not None:
    final_pred_df = final_pred_df.merge(METADATA[TARGET], on='patient_id', how='left', suffixes=('', '_meta'))
final_pred_path = os.path.join(RESULTS_DIR, "test_final_model_predictions.csv")
final_pred_df.to_csv(final_pred_path, index=False)

print(f"Cross-task ({CORPORA[SOURCE]['name']} -> {CORPORA[TARGET]['name']}), "
      f"final model trained on 100% of the training corpus, frozen:")
print(pd.Series(final_target_metrics).to_string())
print()
print(classification_report(target_labels, (final_probs > 0.5).astype(int),
                            target_names=[NEG_NAME[TARGET], 'AD'], digits=4))
print(f"Saved {final_pred_path}")

In [ ]:
# Subgroups of the test set
def parse_bool(x):
    v = str(x).strip().lower()
    if v in {'true', '1', 'yes', 'y', 't'}:
        return True
    if v in {'false', '0', 'no', 'n', 'f'}:
        return False
    return np.nan

SEEN_COL = find_column(final_pred_df, ['seen_in_training', 'seen'])
DX_COL = find_column(final_pred_df, ['original_diagnosis', 'original_dx', 'diagnosis', 'dx'])
if SEEN_COL is None or DX_COL is None:
    print(f"NOTE test metadata columns: {list(METADATA[TARGET].columns) if METADATA[TARGET] is not None else None} "
          f"-> seen column: {SEEN_COL}, diagnosis column: {DX_COL}")

subgroup_masks = {'all test recordings': np.ones(len(final_pred_df), dtype=bool)}
unseen_mask = None
if SEEN_COL is not None:
    seen = final_pred_df[SEEN_COL].map(parse_bool)
    if seen.isna().any():
        print(f"WARNING {int(seen.isna().sum())} test recordings have no usable {SEEN_COL} value -> left out of the seen/unseen rows")
    unseen_mask = (seen == False).values
    subgroup_masks['speaker NOT seen in training'] = unseen_mask
    subgroup_masks['speaker seen in training'] = (seen == True).values
if DX_COL is not None:
    for dx in sorted(final_pred_df[DX_COL].dropna().unique()):
        subgroup_masks[f'original diagnosis = {dx}'] = (final_pred_df[DX_COL] == dx).values
    if SEEN_COL is not None:
        for dx in sorted(final_pred_df[DX_COL].dropna().unique()):
            is_dx = (final_pred_df[DX_COL] == dx).values
            if (is_dx & (seen == True).values).any() and (is_dx & (seen == False).values).any():
                subgroup_masks[f'{dx}, speaker seen in training'] = is_dx & (seen == True).values
                subgroup_masks[f'{dx}, speaker NOT seen in training'] = is_dx & (seen == False).values

subgroup_rows = []
for name, mask in subgroup_masks.items():
    if mask.sum() == 0:
        continue
    sub = final_pred_df[mask]
    sm = compute_metrics(sub['true_label'].values, sub['predicted_probability'].values)
    sens_ci = wilson_ci(sm['tp'], sm['tp'] + sm['fn'])
    spec_ci = wilson_ci(sm['tn'], sm['tn'] + sm['fp'])
    subgroup_rows.append({'subgroup': name, 'n_ad': int(sub['true_label'].sum()),
                          'n_cn': int((sub['true_label'] == 0).sum()), **sm,
                          'sensitivity_ci_low': sens_ci[0], 'sensitivity_ci_high': sens_ci[1],
                          'specificity_ci_low': spec_ci[0], 'specificity_ci_high': spec_ci[1]})
subgroup_df = pd.DataFrame(subgroup_rows).set_index('subgroup')
subgroup_path = os.path.join(RESULTS_DIR, "test_subgroup_metrics.csv")
subgroup_df.to_csv(subgroup_path)
print(f"Saved {subgroup_path}")
subgroup_df.round(4)

In [ ]:
# RQ2 summary and test report
metric_cols = ['balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc', 'f1_ad', 'accuracy', 'pred_ad_rate']

oof_row = f'in-domain {SOURCE} (OOF pooled)'
final_row = f'{SOURCE}->{TARGET} final model (frozen)'
unseen_row = f'{SOURCE}->{TARGET} final model (frozen), unseen speakers only'
summary_rows = [
    {'setting': oof_row, **{m: source_oof_metrics[m] for m in metric_cols}},
    {'setting': f'in-domain {SOURCE} (per-fold mean)', **source_fold_df[metric_cols].mean().to_dict()},
    {'setting': f'in-domain {SOURCE} (per-fold std)', **source_fold_df[metric_cols].std().to_dict()},
    {'setting': final_row, **{m: final_target_metrics[m] for m in metric_cols}},
]
if unseen_mask is not None and unseen_mask.any():
    unseen_metrics = compute_metrics(target_labels[unseen_mask], final_probs[unseen_mask])
    summary_rows.append({'setting': unseen_row, **{m: unseen_metrics[m] for m in metric_cols}})
summary_df = pd.DataFrame(summary_rows).set_index('setting')
summary_df.loc['transfer gap (OOF - final model)'] = summary_df.loc[oof_row] - summary_df.loc[final_row]
if unseen_row in summary_df.index:
    summary_df.loc['transfer gap (OOF - final model, unseen)'] = summary_df.loc[oof_row] - summary_df.loc[unseen_row]

print(f"Final model: {FINAL_EPOCHS} epochs on 100% of {CORPORA[SOURCE]['name']} (CV best epochs: {best_epochs})")
print(f"Test true AD share: {target_labels.mean():.3f}  |  training true AD share: {source_labels.mean():.3f}")
summary_path = os.path.join(RESULTS_DIR, f"rq2_summary_{SOURCE}_to_{TARGET}.csv")
summary_df.to_csv(summary_path)
print(f"Saved {summary_path}")
summary_df.round(4)

In [ ]:
# RQ2 summary and test report (continued)
test_predictions = (final_probs > 0.5).astype(int)
test_accuracy = accuracy_score(target_labels, test_predictions)
test_report = classification_report(
    target_labels, test_predictions, target_names=[NEG_NAME[TARGET], 'AD'], digits=4
)

m = final_target_metrics
majority_baseline = max(target_labels.mean(), 1 - target_labels.mean())
test_summary = (
    f"Independent test on {CORPORA[TARGET]['name']} (final model, {FINAL_EPOCHS} epochs on 100% of {CORPORA[SOURCE]['name']}, frozen, n={m['n']}):\n"
    f"  Balanced accuracy : {m['balanced_accuracy']:.4f}\n"
    f"  Sensitivity (AD)  : {m['sensitivity']:.4f}\n"
    f"  Specificity (CN)  : {m['specificity']:.4f}  (all test CN are MCI)\n"
    f"  ROC-AUC           : {m['roc_auc']:.4f}\n"
    f"  F1 (AD)           : {m['f1_ad']:.4f}\n"
    f"  Accuracy          : {test_accuracy:.4f}  (majority-class baseline: {majority_baseline:.4f})\n"
    f"  Confusion matrix  : TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}\n"
)
if unseen_row in summary_df.index:
    u = summary_df.loc[unseen_row]
    test_summary += (f"  Unseen speakers only (n={int(unseen_mask.sum())}, AD {int(target_labels[unseen_mask].sum())} / "
                     f"CN {int((target_labels[unseen_mask] == 0).sum())}): balanced accuracy {u['balanced_accuracy']:.4f}, "
                     f"sensitivity {u['sensitivity']:.4f}, specificity {u['specificity']:.4f}, ROC-AUC {u['roc_auc']:.4f}\n")

print(test_summary)
print(test_report)

test_results_df = final_pred_df.copy()
results_csv_path = os.path.join(RESULTS_DIR, "test_evaluation.csv")
report_path = os.path.join(RESULTS_DIR, "test_report.txt")
test_results_df.to_csv(results_csv_path, index=False)
with open(report_path, 'w') as f:
    f.write(test_summary + "\n")
    f.write(test_report)
print(f"Saved per-recording results to {results_csv_path}")
print(f"Saved metrics/classification report to {report_path}")
test_results_df

In [ ]:
# 95% confidence intervals
def metrics_with_ci(y_true, probs):
    m = compute_metrics(y_true, probs)
    boot = bootstrap_ci(y_true, probs)
    return {
        'n': m['n'], 'n_ad': int(m['tp'] + m['fn']), 'n_cn': int(m['tn'] + m['fp']),
        'balanced_accuracy': m['balanced_accuracy'], 'balanced_accuracy_ci': boot['balanced_accuracy'],
        'sensitivity': m['sensitivity'], 'sensitivity_ci': wilson_ci(m['tp'], m['tp'] + m['fn']),
        'specificity': m['specificity'], 'specificity_ci': wilson_ci(m['tn'], m['tn'] + m['fp']),
        'roc_auc': m['roc_auc'], 'roc_auc_ci': boot['roc_auc'],
    }

CI_METRICS = ['balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc']
ci_sets = {oof_row: (source_oof_df['true_label'].values, source_oof_df['predicted_probability'].values),
           final_row: (target_labels, final_probs)}
if unseen_mask is not None and unseen_mask.any():
    ci_sets[unseen_row] = (target_labels[unseen_mask], final_probs[unseen_mask])

ci_rows, ci_pretty = [], []
for setting, (y_, p_) in ci_sets.items():
    r = metrics_with_ci(y_, p_)
    row = {'setting': setting, 'n': r['n'], 'n_ad': r['n_ad'], 'n_cn': r['n_cn']}
    pretty = dict(row)
    for k in CI_METRICS:
        lo, hi = r[f'{k}_ci']
        row.update({k: r[k], f'{k}_ci_low': lo, f'{k}_ci_high': hi})
        pretty[k] = f"{r[k]:.3f} [{lo:.3f}, {hi:.3f}]"
    ci_rows.append(row)
    ci_pretty.append(pretty)
ci_df = pd.DataFrame(ci_rows).set_index('setting')
ci_pretty_df = pd.DataFrame(ci_pretty).set_index('setting')

ci_path = os.path.join(RESULTS_DIR, "rq2_confidence_intervals.csv")
ci_pretty_path = os.path.join(RESULTS_DIR, "rq2_metrics_with_ci.csv")
ci_df.to_csv(ci_path)
ci_pretty_df.to_csv(ci_pretty_path)

ci_summary = "95% confidence intervals (Wilson for sensitivity/specificity, stratified bootstrap x"
ci_summary += f"{N_BOOTSTRAP} for balanced accuracy/ROC-AUC):\n"
for setting, r in ci_pretty_df.iterrows():
    ci_summary += f"  {setting} (n={r['n']}, AD {r['n_ad']} / CN {r['n_cn']}):\n"
    for k in CI_METRICS:
        ci_summary += f"    {k:<18}: {r[k]}\n"
print(ci_summary)

with open(report_path, 'w') as fh:
    fh.write(test_summary + "\n")
    fh.write(ci_summary + "\n")
    fh.write(test_report)
print(f"Saved {ci_path}\nSaved {ci_pretty_path}\nUpdated {report_path}")
ci_pretty_df

In [ ]:
# Recording length vs. predicted P(AD)
from scipy.stats import spearmanr, mannwhitneyu

def raw_durations(key):
    frames = []
    for cls in CLASS_LABEL:
        df = pd.read_csv(PATHS[key][f"{cls}_raw_features_csv"], dtype={'patient_id': str})[['patient_id', 'duration']]
        df['label'] = CLASS_LABEL[cls]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

train_dur = raw_durations(SOURCE)
test_dur = final_pred_df[['patient_id', 'true_label', 'predicted_probability']].merge(
    raw_durations(TARGET)[['patient_id', 'duration']], on='patient_id', how='left')
assert test_dur['duration'].notna().all(), "test recordings without a duration (check the acoustic features)"
test_seen = (final_pred_df[SEEN_COL].map(parse_bool) == True).values if SEEN_COL is not None else np.zeros(len(test_dur), bool)

def spearman(x, y):
    if len(x) < 3:
        return float('nan'), float('nan')
    rho, pval = spearmanr(x, y)
    return float(rho), float(pval)

duration_rows = []
for set_name, df in [('train', train_dur), ('test', test_dur)]:
    for lab, name in [(1, 'AD'), (0, 'CN')]:
        d = df.loc[df['label' if set_name == 'train' else 'true_label'] == lab, 'duration']
        duration_rows.append({'set': set_name, 'class': name, 'n': len(d), 'median_s': d.median(),
                              'q25_s': d.quantile(0.25), 'q75_s': d.quantile(0.75)})
duration_table = pd.DataFrame(duration_rows)

u_train = mannwhitneyu(train_dur.loc[train_dur['label'] == 1, 'duration'],
                       train_dur.loc[train_dur['label'] == 0, 'duration'])
rho_all = spearman(test_dur['duration'], test_dur['predicted_probability'])
rho_ad = spearman(*test_dur.loc[test_dur['true_label'] == 1, ['duration', 'predicted_probability']].T.values)
rho_cn = spearman(*test_dur.loc[test_dur['true_label'] == 0, ['duration', 'predicted_probability']].T.values)

duration_summary = (
    "Recording length (raw duration in seconds):\n"
    + duration_table.round(1).to_string(index=False) + "\n"
    f"  Training AD vs CN length: Mann-Whitney U = {u_train.statistic:.1f}, p = {u_train.pvalue:.4f}\n"
    f"  Test, Spearman rho(length, P(AD)): all {rho_all[0]:.3f} (p = {rho_all[1]:.4f}), "
    f"AD only {rho_ad[0]:.3f} (p = {rho_ad[1]:.4f}), CN only {rho_cn[0]:.3f} (p = {rho_cn[1]:.4f})\n"
)
print(duration_summary)

COLORS = {0: '#2a78d6', 1: '#eb6834'}
MARKERS = {0: 'o', 1: '^'}
NAMES = {0: 'CN', 1: 'AD'}
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(11, 4.2))

groups = [('Train CN', train_dur.loc[train_dur['label'] == 0, 'duration'], 0),
          ('Train AD', train_dur.loc[train_dur['label'] == 1, 'duration'], 1),
          ('Test CN (MCI)', test_dur.loc[test_dur['true_label'] == 0, 'duration'], 0),
          ('Test AD', test_dur.loc[test_dur['true_label'] == 1, 'duration'], 1)]
bp = ax_a.boxplot([g[1] for g in groups], widths=0.5, patch_artist=True, showfliers=False,
                  medianprops={'color': '#222222', 'linewidth': 2})
for patch, (_, _, lab) in zip(bp['boxes'], groups):
    patch.set_facecolor(COLORS[lab]); patch.set_alpha(0.35); patch.set_edgecolor(COLORS[lab])
rng_jit = np.random.default_rng(SEED)
for i, (_, d, lab) in enumerate(groups, start=1):
    ax_a.scatter(i + rng_jit.uniform(-0.15, 0.15, len(d)), d, s=12, color=COLORS[lab], marker=MARKERS[lab],
                 alpha=0.7, linewidths=0)
ax_a.set_xticks(range(1, len(groups) + 1))
ax_a.set_xticklabels([f"{g[0]}\n(n={len(g[1])})" for g in groups])
ax_a.set_ylabel('Recording length (s)')
ax_a.set_title('(a) Recording length by set and class')

for lab in (0, 1):
    for seen_flag, face in ((True, 'filled'), (False, 'hollow')):
        sel = (test_dur['true_label'].values == lab) & (test_seen == seen_flag)
        if not sel.any():
            continue
        ax_b.scatter(test_dur.loc[sel, 'duration'], test_dur.loc[sel, 'predicted_probability'], s=40,
                     marker=MARKERS[lab], edgecolors=COLORS[lab], linewidths=1.5,
                     facecolors=COLORS[lab] if face == 'filled' else 'none',
                     label=f"{NAMES[lab]}, {'seen' if seen_flag else 'unseen'} speaker (n={int(sel.sum())})")
ax_b.axhline(0.5, color='#888888', linewidth=1, linestyle='--')
ax_b.text(ax_b.get_xlim()[1], 0.5, ' threshold 0.5', va='center', ha='left', fontsize=8, color='#555555')
ax_b.set_ylim(-0.03, 1.03)
ax_b.set_xlabel('Recording length (s)')
ax_b.set_ylabel('Predicted P(AD)')
ax_b.set_title(f"(b) Test set: P(AD) vs. length (Spearman rho = {rho_all[0]:.2f})")
ax_b.legend(fontsize=8, loc='lower right', frameon=False)

for ax in (ax_a, ax_b):
    ax.grid(True, axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
duration_plot_path = os.path.join(RESULTS_DIR, "rq2_duration_vs_probability.png")
plt.savefig(duration_plot_path, dpi=200, bbox_inches='tight')
plt.savefig(duration_plot_path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()

duration_table.to_csv(os.path.join(RESULTS_DIR, "rq2_duration_by_class.csv"), index=False)
test_dur.to_csv(os.path.join(RESULTS_DIR, "test_duration_vs_probability.csv"), index=False)
with open(report_path, 'w') as fh:
    fh.write(test_summary + "\n")
    fh.write(globals().get('ci_summary', '') + "\n")
    fh.write(duration_summary + "\n")
    fh.write(test_report)
print(f"Saved {duration_plot_path} (+ .pdf)\nUpdated {report_path}")
duration_table.round(1)

In [ ]:
# Transferability estimates: LEEP and NCE
LOGME_REPO = None
LOGME_GIT_URL = "https://github.com/thuml/LogME.git"

def find_logme_repo():
    if LOGME_REPO:
        return os.path.abspath(LOGME_REPO)
    here = os.path.abspath(os.getcwd())
    while True:
        candidate = os.path.join(here, "../../LogME")
        if os.path.isfile(os.path.join(candidate, "LEEP.py")):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

LOGME_REPO = find_logme_repo()
if LOGME_REPO is None:
    LOGME_REPO = os.path.abspath(os.path.join("..", "..", "../../LogME"))
    print(f"LogME repo not found -> cloning {LOGME_GIT_URL} into {LOGME_REPO}")
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", LOGME_GIT_URL, LOGME_REPO], check=True)
for f in ("LEEP.py", "NCE.py"):
    if not os.path.isfile(os.path.join(LOGME_REPO, f)):
        raise FileNotFoundError(f"{f} not found in {LOGME_REPO}. Copy the LogME folder from the project root "
                                f"to this machine, or set LOGME_REPO to the folder that contains it")
print(f"Using LEEP / NCE from: {LOGME_REPO}")
if LOGME_REPO not in sys.path:
    sys.path.insert(0, LOGME_REPO)
from LEEP import LEEP
from NCE import NCE

def label_entropy(y):
    prior = np.bincount(np.asarray(y, dtype=int), minlength=2) / len(y)
    prior = prior[prior > 0]
    return float(-(prior * np.log(prior)).sum())

def transferability_scores(y, probs, threshold=0.5):
    y = np.asarray(y, dtype=int)
    probs = np.asarray(probs, dtype=np.float64)
    dummy_dist = np.stack([1.0 - probs, probs], axis=1)
    dummy_label = (probs > threshold).astype(int)
    h_y = label_entropy(y)
    leep = float(LEEP(dummy_dist, y))
    nce = float(NCE(dummy_label, y))
    return {
        'leep': leep,
        'nce': nce,
        'label_entropy': h_y,
        'leep_minus_prior': leep + h_y,
        'nce_normalized': 1.0 + nce / h_y if h_y > 0 else float('nan'),
    }

TRANSFER_COLS = ['leep', 'nce', 'label_entropy', 'leep_minus_prior', 'nce_normalized']

In [ ]:
# Transferability estimates: LEEP and NCE (continued)
source_oof_scores = transferability_scores(source_oof_df['true_label'].values, source_oof_df['predicted_probability'].values)
per_fold_transfer_df = pd.DataFrame([
    {'fold': k, 'n': len(g), **transferability_scores(g['true_label'].values, g['predicted_probability'].values)}
    for k, g in source_oof_df.groupby('fold')
])

final_scores = transferability_scores(target_labels, final_probs)

transfer_rows = {
    oof_row: source_oof_scores,
    f'in-domain {SOURCE} (per-fold mean)': per_fold_transfer_df[TRANSFER_COLS].mean().to_dict(),
    f'in-domain {SOURCE} (per-fold std)': per_fold_transfer_df[TRANSFER_COLS].std().to_dict(),
    final_row: final_scores,
}
if unseen_row in summary_df.index and len(np.unique(target_labels[unseen_mask])) == 2:
    transfer_rows[unseen_row] = transferability_scores(target_labels[unseen_mask], final_probs[unseen_mask])

transfer_df = pd.DataFrame.from_dict(transfer_rows, orient='index')[TRANSFER_COLS]
transfer_df.index.name = 'setting'
transfer_df.loc['transfer gap (OOF - final model)'] = transfer_df.loc[oof_row] - transfer_df.loc[final_row]
if unseen_row in transfer_df.index:
    transfer_df.loc['transfer gap (OOF - final model, unseen)'] = transfer_df.loc[oof_row] - transfer_df.loc[unseen_row]

transfer_path = os.path.join(RESULTS_DIR, f"transferability_scores_{SOURCE}_to_{TARGET}.csv")
per_fold_path = os.path.join(RESULTS_DIR, f"transferability_per_fold_{SOURCE}.csv")
transfer_df.to_csv(transfer_path)
per_fold_transfer_df.to_csv(per_fold_path, index=False)

print(f"Transferability {CORPORA[SOURCE]['name']} -> {CORPORA[TARGET]['name']} "
      f"(chance level: LEEP = NCE = -H(Y); nce_normalized = leep_minus_prior = 0)")
print(f"Test H(Y) = {final_scores['label_entropy']:.4f}  |  training H(Y) = {source_oof_scores['label_entropy']:.4f}")
print(f"Saved {transfer_path}\nSaved {per_fold_path}")
transfer_df.round(4)

In [ ]:
# Transferability estimates: LEEP and NCE (continued)
summary_df = summary_df.drop(columns=[c for c in TRANSFER_COLS if c in summary_df.columns]).join(transfer_df)
summary_df.to_csv(summary_path)
print(f"Updated {summary_path} with {TRANSFER_COLS}")

f = final_scores
transfer_summary = (
    f"Transferability of the frozen final model to {CORPORA[TARGET]['name']} (n={len(target_labels)}):\n"
    f"  LEEP              : {f['leep']:.4f}  (chance -H(Y) = {-f['label_entropy']:.4f}, gain over prior {f['leep_minus_prior']:.4f})\n"
    f"  NCE               : {f['nce']:.4f}  (chance -H(Y) = {-f['label_entropy']:.4f}, normalized {f['nce_normalized']:.4f})\n"
    f"  In-domain {SOURCE} OOF: LEEP {source_oof_scores['leep']:.4f}, NCE {source_oof_scores['nce']:.4f}, "
    f"normalized NCE {source_oof_scores['nce_normalized']:.4f}\n"
)
print(test_summary)
print(transfer_summary)
with open(report_path, 'w') as fh:
    fh.write(test_summary + "\n")
    fh.write(globals().get('ci_summary', '') + "\n")
    fh.write(globals().get('duration_summary', '') + "\n")
    fh.write(transfer_summary + "\n")
    fh.write(test_report)
print(f"Updated {report_path}")
summary_df.round(4)